# FairFace Bias Analysis

This notebook trains a couple of transfer-learning models on the FairFace dataset, selects the best overall model on the validation split, and reports how it performs across racial groups. Visualizations are saved to `outputs/`.

**Quick start**
1. Ensure the dataset lives under `FairFace/` (with `train/`, `val/`, and label CSVs).
2. Install deps once in your env: `pip install -r requirements.txt`.
3. Run cells top-to-bottom. Adjust `MAX_TRAIN_SAMPLES`/`MAX_VAL_SAMPLES` for faster dry-runs.

In [1]:
import random
from pathlib import Path
from typing import Dict, List, Tuple

import matplotlib.pyplot as plt
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision.models as models
import torchvision.transforms as T
from PIL import Image
from torch.utils.data import DataLoader, Dataset

torch.__version__

'2.9.1+cpu'

In [2]:
# Configuration
DATA_ROOT = Path("FairFace")
TRAIN_CSV = DATA_ROOT / "train_labels.csv"
VAL_CSV = DATA_ROOT / "val_labels.csv"
OUTPUT_DIR = Path("outputs")

# Models to compare
MODEL_NAMES = ["resnet18", "mobilenet_v3_small"]

# Training hyperparams (tweak to taste)
EPOCHS = 1
BATCH_SIZE = 32
IMAGE_SIZE = 224
LR = 1e-3
NUM_WORKERS = 4
FINETUNE_BACKBONE = False  # True will unfreeze backbone (slower, usually better)
SEED = 42

# Set to 0 or None to use all samples; keep small for quick iteration
MAX_TRAIN_SAMPLES = 5000
MAX_VAL_SAMPLES = 2000

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

device(type='cpu')

In [3]:
def set_seed(seed: int) -> None:
    random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


class FairFaceDataset(Dataset):
    def __init__(self, root: Path, csv_path: Path, transform: T.Compose, race_to_idx: Dict[str, int], max_samples: int | None = None):
        self.root = root
        self.transform = transform
        self.race_to_idx = race_to_idx
        df = pd.read_csv(csv_path)
        if max_samples is not None and max_samples > 0:
            df = df.sample(n=min(max_samples, len(df)), random_state=42)
        self.items = df[["file", "race"]].values.tolist()

    def __len__(self):
        return len(self.items)

    def __getitem__(self, idx):
        rel_path, race_label = self.items[idx]
        img_path = self.root / rel_path
        with Image.open(img_path) as img:
            img = img.convert("RGB")
        label_idx = self.race_to_idx[race_label]
        return self.transform(img), label_idx, race_label


def build_transforms(image_size: int) -> Tuple[T.Compose, T.Compose]:
    train_tfms = T.Compose([
        T.Resize((image_size, image_size)),
        T.RandomHorizontalFlip(p=0.5),
        T.ColorJitter(brightness=0.1, contrast=0.1, saturation=0.1),
        T.ToTensor(),
        T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ])
    eval_tfms = T.Compose([
        T.Resize((image_size, image_size)),
        T.ToTensor(),
        T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ])
    return train_tfms, eval_tfms


def create_model(name: str, num_classes: int, finetune_backbone: bool):
    name = name.lower()
    if name == "resnet18":
        weights = models.ResNet18_Weights.IMAGENET1K_V1
        model = models.resnet18(weights=weights)
        in_features = model.fc.in_features
        model.fc = nn.Linear(in_features, num_classes)
        head_params = list(model.fc.parameters())
        backbone_params = [p for n, p in model.named_parameters() if not n.startswith("fc.")]
    elif name == "mobilenet_v3_small":
        weights = models.MobileNet_V3_Small_Weights.IMAGENET1K_V1
        model = models.mobilenet_v3_small(weights=weights)
        in_features = model.classifier[3].in_features
        model.classifier[3] = nn.Linear(in_features, num_classes)
        head_params = list(model.classifier[3].parameters())
        backbone_params = [p for n, p in model.named_parameters() if not n.startswith("classifier.3")]
    else:
        raise ValueError(f"Unsupported model name: {name}")

    params_to_train: List[torch.nn.Parameter] = head_params
    if finetune_backbone:
        params_to_train = list(model.parameters())
    else:
        for p in backbone_params:
            p.requires_grad = False
    return model, params_to_train


In [4]:
def train_one_epoch(model, loader, criterion, optimizer, device):
    model.train()
    running_loss = 0.0
    total = 0
    for imgs, labels, _ in loader:
        imgs = imgs.to(device)
        labels = labels.to(device)
        optimizer.zero_grad()
        outputs = model(imgs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item() * labels.size(0)
        total += labels.size(0)
    return running_loss / max(total, 1)


@torch.no_grad()
def evaluate(model, loader, device, idx_to_race):
    model.eval()
    correct = 0
    total = 0
    per_race: Dict[str, Dict[str, int]] = {}
    for imgs, labels, race_labels in loader:
        imgs = imgs.to(device)
        labels = labels.to(device)
        outputs = model(imgs)
        preds = outputs.argmax(dim=1)
        correct += (preds == labels).sum().item()
        total += labels.size(0)
        for pred_idx, target_idx, race_label in zip(preds.cpu().tolist(), labels.cpu().tolist(), race_labels):
            race = idx_to_race[target_idx]
            stats = per_race.setdefault(race, {"correct": 0, "total": 0})
            stats["total"] += 1
            if pred_idx == target_idx:
                stats["correct"] += 1
    overall_acc = correct / max(total, 1)
    return overall_acc, per_race


def plot_distribution(df: pd.DataFrame, out_path: Path) -> None:
    counts = df["race"].value_counts().sort_index()
    plt.figure(figsize=(8, 4))
    counts.plot(kind="bar", color="#4C72B0")
    plt.title("Validation Set Race Distribution")
    plt.xlabel("Race")
    plt.ylabel("Count")
    plt.tight_layout()
    out_path.parent.mkdir(parents=True, exist_ok=True)
    plt.savefig(out_path)
    plt.close()


def plot_performance(per_race: Dict[str, Dict[str, int]], out_path: Path) -> None:
    races = sorted(per_race.keys())
    accuracies = [per_race[r]["correct"] / max(per_race[r]["total"], 1) for r in races]
    plt.figure(figsize=(8, 4))
    plt.bar(races, accuracies, color="#55A868")
    plt.title("Per-race Accuracy (Best Model)")
    plt.xlabel("Race")
    plt.ylabel("Accuracy")
    plt.ylim(0, 1)
    plt.tight_layout()
    out_path.parent.mkdir(parents=True, exist_ok=True)
    plt.savefig(out_path)
    plt.close()


In [6]:
# Prepare data, loaders, and metadata
set_seed(SEED)
train_df = pd.read_csv(TRAIN_CSV)
val_df = pd.read_csv(VAL_CSV)
class_names = sorted(train_df["race"].unique())
race_to_idx = {race: i for i, race in enumerate(class_names)}
idx_to_race = {i: race for race, i in race_to_idx.items()}

train_tfms, eval_tfms = build_transforms(IMAGE_SIZE)
train_ds = FairFaceDataset(DATA_ROOT, TRAIN_CSV, transform=train_tfms, race_to_idx=race_to_idx, max_samples=MAX_TRAIN_SAMPLES)
val_ds = FairFaceDataset(DATA_ROOT, VAL_CSV, transform=eval_tfms, race_to_idx=race_to_idx, max_samples=MAX_VAL_SAMPLES)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS, pin_memory=True)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=True)

len(train_ds), len(val_ds), class_names

(5000,
 2000,
 ['Black',
  'East Asian',
  'Indian',
  'Latino_Hispanic',
  'Middle Eastern',
  'Southeast Asian',
  'White'])

In [7]:
# Visualize validation race distribution
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
plot_distribution(val_df, OUTPUT_DIR / "val_race_distribution.png")
pd.read_csv(VAL_CSV).race.value_counts()

race
White              2085
Latino_Hispanic    1623
Black              1556
East Asian         1550
Indian             1516
Southeast Asian    1415
Middle Eastern     1209
Name: count, dtype: int64

In [ ]:
# Train/eval multiple models and pick the best overall
metrics: List[Dict] = []
best_model_name = None
best_model_state = None
best_performance = -1.0
best_per_race = None

for model_name in MODEL_NAMES:
    model, params_to_train = create_model(model_name, num_classes=len(class_names), finetune_backbone=FINETUNE_BACKBONE)
    model.to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(params_to_train, lr=LR)

    for _ in range(EPOCHS):
        train_loss = train_one_epoch(model, train_loader, criterion, optimizer, device)
    overall_acc, per_race = evaluate(model, val_loader, device, idx_to_race)
    metrics.append({"model": model_name, "overall_accuracy": overall_acc, "per_race": per_race})
    if overall_acc > best_performance:
        best_performance = overall_acc
        best_model_name = model_name
        best_model_state = model.state_dict()
        best_per_race = per_race

metrics

c:\Users\varun\OneDrive\Desktop\Bigdata case study\myenv\Lib\site-packages\torch\utils\data\dataloader.py:668: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


In [ ]:
# Persist artifacts and visualize per-race performance of the best model
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
pd.DataFrame([{"model": m["model"], "overall_accuracy": m["overall_accuracy"]} for m in metrics]).to_json(OUTPUT_DIR / "model_metrics.json", orient="records", indent=2)

if best_model_state is not None:
    torch.save({"model_name": best_model_name, "state_dict": best_model_state, "class_names": class_names}, OUTPUT_DIR / f"best_model_{best_model_name}.pth")

per_race_records = []
for race, stats in (best_per_race or {}).items():
    per_race_records.append({
        "race": race,
        "correct": stats["correct"],
        "total": stats["total"],
        "accuracy": stats["correct"] / max(stats["total"], 1),
        "model": best_model_name,
    })
per_race_df = pd.DataFrame(per_race_records)
per_race_df.to_csv(OUTPUT_DIR / "best_model_per_race.csv", index=False)

if best_per_race:
    plot_performance(best_per_race, OUTPUT_DIR / "best_model_per_race.png")

best_model_name, best_performance, per_race_df